In [1]:
# Install zstd (Required for extraction) and LangChain libraries
!sudo apt-get update && sudo apt-get install -y zstd
!pip install langchain-ollama langchain-community
!sudo systemctl enable --now ollama
!pip install langchain langchain-ollama sqlalchemy ollama

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [2]:
# 1. Install Ollama and the Python library
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q

import subprocess
import time
import threading
import ollama

# 2. Start Ollama server in the background
def run_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5)  # Give the server a moment to warm up

# 3. Pull a odel (Llama 3.2 is ~2GB)
print("Downloading model...")
ollama.pull('llama3.2')

# 4. Ask a simple question
print("\n--- Model Response ---")
response = ollama.chat(model='llama3.2', messages=[
    {'role': 'user', 'content': 'What is the most interesting fact about space?'}
])

print(response['message']['content'])

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

--- Model Response ---
There are so many fascinating facts about space, it's hard to choose just one. However, here's a mind-blowing one:

**The Universe is Still Expanding**

In the 1920s, astronomer Edwin Hubble made a groundbreaking discovery that revolutionized our understanding of the universe. He observed that the light coming from distant galaxies was shifted towards the red end of the spectrum, a phenomenon known as redshift. This meant that those galaxies were moving away from us!

But here's the kicker: the farther away these galaxies

In [3]:
import sqlite3
import subprocess
import time

from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_ollama import ChatOllama

# -----------------------------
# START OLLAMA SERVER
# -----------------------------
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# -----------------------------
# CREATE DATABASE
# -----------------------------
conn = sqlite3.connect("school.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    marks INTEGER
)
""")

cursor.execute("DELETE FROM students")

cursor.executemany(
    "INSERT INTO students (name, marks) VALUES (?, ?)",
    [("Alice", 85), ("Bob", 78), ("Charlie", 92)]
)

conn.commit()
conn.close()

# -----------------------------
# CONNECT DATABASE
# -----------------------------
db = SQLDatabase.from_uri("sqlite:///school.db")
executor = QuerySQLDataBaseTool(db=db)

# -----------------------------
# LOAD MODEL
# -----------------------------
llm = ChatOllama(model="llama3.2")

# -----------------------------
# NATURAL LANGUAGE → SQL (MANUAL PROMPT)
# -----------------------------
question = "Who scored the highest marks?"

prompt = f"""
You are an expert SQL assistant.

Convert this question into a valid SQLite query.

Table name: students
Columns: id, name, marks

Question: {question}

Only return SQL query, nothing else.
"""

sql_query = llm.invoke(prompt).content.strip()

print("Generated SQL:\n", sql_query)

# -----------------------------
# EXECUTE QUERY
# -----------------------------
result = executor.invoke(sql_query)

print("\nFinal Answer:\n", result)


/tmp/ipykernel_7306/3293583285.py:43: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  executor = QuerySQLDataBaseTool(db=db)


Generated SQL:
 SELECT name FROM students ORDER BY marks DESC LIMIT 1

Final Answer:
 [('Charlie',)]
